# NC Field Landsat NDVI Analysis

**Date:** 2026-04-07  
**Data Source:** Landsat 7 Collection 2 Level-2 (LE07_L2SP_016035_20231104_02_T1) via Microsoft Planetary Computer STAC API

## Summary

This notebook analyzes satellite imagery for the largest agricultural field in North Carolina from the field boundaries dataset. The analysis includes:

1. **Spectral Band Extraction** - Downloaded and clipped 6 Landsat bands (Blue, Green, Red, NIR, SWIR1, SWIR2) to the field boundary
2. **NDVI Calculation** - Computed Normalized Difference Vegetation Index using Red (B4) and Near-Infrared (B5) bands
3. **Visualization** - Created color maps for each band and NDVI

The largest field (768 acres, Guilford County) was analyzed using a Landsat 7 scene from November 4, 2023 with 0% cloud cover.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import Image, display, Markdown

DATA_DIR = Path("/workspaces/my-farm-advisor/data/workspace")
OUTPUT_DIR = DATA_DIR / "output" / "assignment-05"

## 1. Largest Field Identified

From the NC field boundaries dataset (23 fields), the largest field was identified:

In [ ]:
fields = gpd.read_file(DATA_DIR / "NC_field_boundaries_EPSG4326_2026-04-01.geojson")
largest = fields.loc[fields['area_acres'].idxmax()]

print(f"Field ID: {largest['field_id']}")
print(f"County: {largest['county_name']}")
print(f"Area: {largest['area_acres']:.2f} acres")
print(f"Crop Type: {largest['crop_name']}")

## 2. Spectral Band Images

Six Landsat bands were extracted and visualized with appropriate color maps:

In [ ]:
band_info = [
    ("B2_Blue", "Blue Band (B2) - Blues colormap", OUTPUT_DIR / "osm-260949778_B2_Blue_color.png"),
    ("B3_Green", "Green Band (B3) - Greens colormap", OUTPUT_DIR / "osm-260949778_B3_Green_color.png"),
    ("B4_Red", "Red Band (B4) - Reds colormap", OUTPUT_DIR / "osm-260949778_B4_Red_color.png"),
    ("B5_NIR", "Near-Infrared Band (B5) - hot_r colormap", OUTPUT_DIR / "osm-260949778_B5_NIR_color.png"),
    ("B6_SWIR1", "SWIR1 Band (B6) - inferno colormap", OUTPUT_DIR / "osm-260949778_B6_SWIR1_color.png"),
    ("B7_SWIR2", "SWIR2 Band (B7) - viridis colormap", OUTPUT_DIR / "osm-260949778_B7_SWIR2_color.png")
]

for band_key, desc, path in band_info:
    display(Markdown(f"### {desc}"))
    display(Image(filename=str(path), width=600))

## 3. Band Statistics Summary

Pixel values scaled 0-255 (derived from surface reflectance / 10000 * 255):

In [ ]:
band_summary = pd.read_csv(OUTPUT_DIR / "band_summary.csv")
print(band_summary.to_string(index=False))

## 4. NDVI Calculation

NDVI (Normalized Difference Vegetation Index) is calculated using the formula:

```
NDVI = (NIR - Red) / (NIR + Red)
```

Where:
- **NIR** = Band 5 (Near-Infrared, B5)
- **Red** = Band 4 (Red, B4)

Raw surface reflectance values were used from the clipped Landsat bands.

In [ ]:
ndvi_summary = pd.read_csv(OUTPUT_DIR / "ndvi_summary.csv")
print(ndvi_summary.to_string(index=False))

## 5. NDVI Visualization

In [ ]:
display(Markdown("### NDVI Color Map (RdYlGn colormap)"))
display(Image(filename=str(OUTPUT_DIR / "osm-260949778_NDVI_color.png"), width=600))

## 6. NDVI Interpretation

| NDVI Value | Interpretation |
|------------|----------------|
| -1.0 to 0 | Water / non-vegetative |
| 0 to 0.2 | Bare soil / rocks |
| 0.2 to 0.5 | Sparse vegetation |
| 0.5 to 1.0 | Dense green vegetation |

**Result:** Mean NDVI of 0.24 indicates sparse to moderate vegetation, consistent with late fall (November) conditions. The scene was captured on November 4, 2023.

## 7. Scripts Used

The following Python scripts were used to generate the outputs:

| Script | Description |
|--------|-------------|
| `generate_bands.py` | Downloads Landsat bands from Planetary Computer STAC API and clips to field boundary |
| `generate_final.py` | Scales band values to 0-255 and reprojects to EPSG:4326 |
| `generate_color_images.py` | Creates color visualization PNGs for each spectral band |
| `generate_ndvi.py` | Calculates NDVI from Red and NIR bands, creates NDVI visualization |

All scripts are saved in `data/workspace/scripts/`.